In [ ]:
import numpy as np
from scipy.stats import chi2
%matplotlib widget
import matplotlib.pyplot as plt
import mtt

In [ ]:
# Map
n = 100
q = .2
r = 1.
model = mtt.Model.CVM(r=r, q=q)
PS = .99
radius = 150.
rng = mtt.get_rng(seed=44)
init_target = 10.

tmap = mtt.scenario_random_spawn(
    ndat=n, model=model, lambda_births=0.1, lambda_init_births=init_target, radius=radius, PS=PS, init_vel_var=1., rng=rng
)

# Radar
radar_seed = 69
lambd = 1e-4
PD = .9
radar = mtt.Radar(lambd=lambd, det_prob=PD, radius=radius, gate_by_rad=True, seed=radar_seed)

# Tracker
PG = .9999
truncation_thr = 1e-5
merge_thr = 6
max_comp = 250
conf_thr = 0.5

n_std = chi2.ppf(PG, df=2) if PG < 1. else 20


# Spawning
dim = model.state_dim
gm = mtt.GaussianMixture(dim)
gm0 = mtt.GaussianMixture(dim=4)

spacing = 25
pos_var = spacing**2
vel_var = 1.0
coords = np.arange(-radius, radius + spacing, spacing)
birth_w = 0.005 / 10
birth_w0 = 0.5*init_target / len(coords)**2 / 0.8
P = np.diag([pos_var, pos_var, vel_var, vel_var]).astype(np.float64)

for x in coords:
    for y in coords:
        if np.hypot(x, y) <= radius * 1.1:
            M = np.array([x, y, 0.0, 0.0], dtype=np.float64)
            gm.push(birth_w, M, P)
            gm0.push(birth_w0, M, P)

In [ ]:
tracker = mtt.PhdTracker(birth_mixture=gm, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=PD, PS=PS, clutter_int=lambd, PG=PG,
                         trunc_thr=truncation_thr, merge_thr=merge_thr, max_components=max_comp, conf_thr=conf_thr, birth_mixture0=gm0
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std},
        'ucov_ellipse': {'alpha':0.5, 'facecolor':'green', 'edgecolor':'green', 'label':'unconf. cov. ell.', 'n_std':n_std},
        'traj_arrow': {'closed':True, 'facecolor':'black', 'edgecolor':'none', 'zorder':10, 'arrow_scale':4.},
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=inner_airports, edge_airports=edge_airports)

# mtt.make_gif(fig, tmap, radar, tracker, plotter, "gifs/phd.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, tracker, plotter, 'PHD', calc_gospa=True, gospa_c=10., gospa_p=2.)

In [ ]:
sim.plot_gospa(rms=False, cols=2, ignore_t0=True);